In [1]:
# ==============================================================================
# DEEPONET COMPLET — Physics-Constrained Deep Operator Network
# ANEO Wind Field Reconstruction — Ines Rouatbi — ISI/ESPRIT — Juillet 2026
# 8 blocs — copier chaque bloc dans une cellule Colab separee
# ==============================================================================

# ==============================================================================
# BLOC 1 — Mount + Imports + Paths + Recreation tenseurs
# ==============================================================================
from google.colab import drive
drive.mount('/content/drive')

import warnings, json, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    total_gpu = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {total_gpu:.1f} GB")
else:
    print("No GPU — Runtime -> Change runtime type -> T4 GPU")
print("Imports OK")

DRIVE_DIR          = Path("/content/drive/MyDrive")
BASE_DIR           = Path("/content/aneo_work")
DATA_DIR           = DRIVE_DIR / "data_4x"
OUT_DIR            = BASE_DIR  / "aneo_output"
MODEL_DIR          = BASE_DIR  / "aneo_models"
TENSOR_DIR         = BASE_DIR  / "aneo_tensors"
FIG_DIR            = BASE_DIR  / "aneo_figures"
DEEPONET_MODEL_DIR = DRIVE_DIR / "aneo_models_deeponet"

for d in [OUT_DIR, MODEL_DIR, TENSOR_DIR, FIG_DIR, DEEPONET_MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"split.json     : {(TENSOR_DIR/'split.json').exists()}")
print(f"norm_stats.csv : {(OUT_DIR/'normalization_stats.csv').exists()}")
print(f"grid_features  : {(TENSOR_DIR/'grid_features.npy').exists()}")
print(f"Tenseurs .npy  : {len(list(TENSOR_DIR.glob('*.npy')))}")

if not (TENSOR_DIR / "split.json").exists():
    print("Recreation tenseurs depuis CSV Drive...")
    obs_files = sorted(DATA_DIR.glob("aneo_observations_day_*.csv"))
    print(f"CSV : {len(obs_files)}")

    grid = pd.read_csv(DATA_DIR / "aneo_grid_points.csv")
    grid = grid.sort_values(['iy','ix']).reset_index(drop=True)
    NX_g, NY_g = 1000, 500

    land_use_vals = sorted(grid['land_use_code'].dropna().unique())
    land_use_map  = {v: i for i, v in enumerate(land_use_vals)}
    grid['land_use_enc'] = grid['land_use_code'].map(
        land_use_map).fillna(0).astype(np.float32)
    lu_mean = float(grid['land_use_enc'].mean())
    lu_std  = float(grid['land_use_enc'].std()) + 1e-8
    grid['land_use_enc_norm'] = (grid['land_use_enc'] - lu_mean) / lu_std

    OBS_COLS = ['point_id','day_index','hour_utc',
                'temperature_c','air_density_kg_m3',
                'relative_humidity_pct','pressure_pa',
                'wind_u_m_s','wind_v_m_s','wind_w_m_s']
    NUMERIC_INPUT_COLS = ['temperature_c','air_density_kg_m3',
                          'relative_humidity_pct','pressure_pa',
                          'wind_u_m_s','wind_v_m_s','wind_w_m_s']
    TARGET_COLS = ['wind_u_m_s','wind_v_m_s','wind_w_m_s']
    DAY_COL     = 'day_index'
    HOUR_COL    = 'hour_utc'
    N_INPUT_CH  = len(NUMERIC_INPUT_COLS) + 1

    sum_dict  = {c: 0.0 for c in NUMERIC_INPUT_COLS}
    sum2_dict = {c: 0.0 for c in NUMERIC_INPUT_COLS}
    count = 0; days_set = set(); hours_set = set()

    for i, f in enumerate(obs_files):
        df = pd.read_csv(f, usecols=OBS_COLS)
        df = df.dropna(subset=[DAY_COL, HOUR_COL])
        for col in NUMERIC_INPUT_COLS:
            vals = df[col].values.astype(np.float64)
            sum_dict[col]  += vals.sum()
            sum2_dict[col] += (vals**2).sum()
        days_set.update(df[DAY_COL].unique())
        hours_set.update(df[HOUR_COL].unique())
        count += len(df)
        if (i+1) % 10 == 0:
            print(f"  {i+1}/{len(obs_files)} fichiers...")

    mean_dict = {c: sum_dict[c]/count for c in sum_dict}
    std_dict  = {c: float(np.sqrt(max(sum2_dict[c]/count - mean_dict[c]**2, 0)+1e-8))
                 for c in sum_dict}
    days  = sorted([int(d) for d in days_set])
    hours = sorted([int(h) for h in hours_set])
    print(f"Jours:{days}  Heures:{hours}")

    pd.DataFrame({col: {'mean': mean_dict[col], 'std': std_dict[col]}
                  for col in NUMERIC_INPUT_COLS}).T.to_csv(
        OUT_DIR / "normalization_stats.csv")

    print("Creation .npy...")
    n_created    = 0
    grid_indexed = grid.set_index('point_id')
    input_norm_cols  = [c+'_norm' for c in NUMERIC_INPUT_COLS] + ['land_use_enc_norm']
    target_norm_cols = [c+'_norm' for c in TARGET_COLS]

    for i, f in enumerate(obs_files):
        df = pd.read_csv(f, usecols=OBS_COLS)
        df = df.dropna(subset=[DAY_COL, HOUR_COL])
        df[DAY_COL]  = df[DAY_COL].astype(int)
        df[HOUR_COL] = df[HOUR_COL].astype(int)
        df = df.merge(grid_indexed[['ix','iy','land_use_enc_norm']],
                      left_on='point_id', right_index=True, how='left')
        for col in NUMERIC_INPUT_COLS:
            df[col+'_norm'] = ((df[col]-mean_dict[col])/std_dict[col]).astype(np.float32)
        for col in TARGET_COLS:
            df[col+'_norm'] = ((df[col]-mean_dict[col])/std_dict[col]).astype(np.float32)
        for day in df[DAY_COL].unique():
            for hour in df[HOUR_COL].unique():
                snap = df[(df[DAY_COL]==day)&(df[HOUR_COL]==hour)].sort_values(['iy','ix'])
                xp = TENSOR_DIR / f"day_{day:03d}_h{hour:02d}_X.npy"
                yp = TENSOR_DIR / f"day_{day:03d}_h{hour:02d}_Y.npy"
                if xp.exists(): n_created += 1; continue
                if len(snap) != NX_g * NY_g: continue
                np.save(xp, snap[input_norm_cols].values.reshape(
                    NY_g, NX_g, N_INPUT_CH).astype(np.float32))
                np.save(yp, snap[target_norm_cols].values.reshape(
                    NY_g, NX_g, len(TARGET_COLS)).astype(np.float32))
                n_created += 1
        gc.collect()
        print(f"  [{i+1}/{len(obs_files)}] {f.name} — {n_created} snapshots")

    def norm_feat(arr): return (arr - arr.mean()) / (arr.std() + 1e-8)
    np.save(TENSOR_DIR / "grid_features.npy", np.stack([
        norm_feat(grid['z_terrain_m'].values).reshape(NY_g, NX_g),
        norm_feat(grid['z_rel_m'].values).reshape(NY_g, NX_g),
        norm_feat(grid['z0_m'].values).reshape(NY_g, NX_g),
        norm_feat(grid['slope_deg'].values).reshape(NY_g, NX_g),
        np.sin(np.deg2rad(grid['aspect_deg'].values)).reshape(NY_g, NX_g),
        np.cos(np.deg2rad(grid['aspect_deg'].values)).reshape(NY_g, NX_g),
        norm_feat(grid['x_m'].values).reshape(NY_g, NX_g),
        norm_feat(grid['y_m'].values).reshape(NY_g, NX_g),
    ], axis=-1).astype(np.float32))

    n_days    = len(days)
    val_days  = [days[i] for i in range(4, n_days, 5)]
    test_days = [days[i] for i in range(2, n_days, 5)]
    train_days = [d for d in days if d not in val_days and d not in test_days]

    with open(TENSOR_DIR / "split.json", "w") as fj:
        json.dump({
            "train_days": train_days, "val_days": val_days, "test_days": test_days,
            "heures": list(hours), "nx": NX_g, "ny": NY_g,
            "n_input": N_INPUT_CH, "n_target": len(TARGET_COLS),
            "input_cols": input_norm_cols, "target_cols": TARGET_COLS,
            "day_col": DAY_COL, "hour_col": HOUR_COL,
            "land_use_map": land_use_map, "lu_mean": lu_mean, "lu_std": lu_std,
            "mean_dict": mean_dict, "std_dict": std_dict,
        }, fj, indent=2)

    print(f"Train:{len(train_days)}j  Val:{len(val_days)}j  Test:{len(test_days)}j")
    print(f"Tenseurs:{n_created}  Pipeline OK")
else:
    print(f"Tenseurs deja presents : {len(list(TENSOR_DIR.glob('*.npy')))} fichiers")


# ==============================================================================
# BLOC 2 — Config + Dataset + Dataloaders
# ==============================================================================
with open(TENSOR_DIR / "split.json", "r") as f:
    split = json.load(f)

TRAIN_DAYS = split["train_days"]
VAL_DAYS   = split["val_days"]
TEST_DAYS  = split["test_days"]
HEURES     = split["heures"]
NX         = split["nx"]
NY         = split["ny"]
N_INPUT    = split["n_input"]
N_TARGET   = split["n_target"]

norm_stats = pd.read_csv(OUT_DIR / "normalization_stats.csv", index_col=0)
STD_U = norm_stats.loc["wind_u_m_s", "std"]
STD_V = norm_stats.loc["wind_v_m_s", "std"]
STD_W = norm_stats.loc["wind_w_m_s", "std"]
MU_U  = norm_stats.loc["wind_u_m_s", "mean"]
MU_V  = norm_stats.loc["wind_v_m_s", "mean"]
MU_W  = norm_stats.loc["wind_w_m_s", "mean"]

mean_dict = split.get("mean_dict", {})
std_dict  = split.get("std_dict",  {})
MU_RHO  = float(mean_dict.get("air_density_kg_m3", 1.154))
STD_RHO = float(std_dict.get("air_density_kg_m3",  0.021))

N_GRID_FEATURES     = 8
N_TEMPORAL_FEATURES = 2
N_INPUT_EFFECTIVE   = N_INPUT + N_GRID_FEATURES + N_TEMPORAL_FEATURES  # 18

print(f"Train : {len(TRAIN_DAYS)}j -> {TRAIN_DAYS}")
print(f"Val   : {len(VAL_DAYS)}j -> {VAL_DAYS}")
print(f"Test  : {len(TEST_DAYS)}j -> {TEST_DAYS}")
print(f"N_INPUT_EFFECTIVE : {N_INPUT_EFFECTIVE}")
print(f"rho : mu={MU_RHO:.4f}  sigma={STD_RHO:.4f}")
print("Config OK")


class ANEODataset(Dataset):
    def __init__(self, days, heures, tensor_dir, augment=False):
        self.tensor_dir = Path(tensor_dir)
        self.augment    = augment
        self.samples    = []
        for day in days:
            for h in heures:
                xp = self.tensor_dir / f"day_{day:03d}_h{h:02d}_X.npy"
                yp = self.tensor_dir / f"day_{day:03d}_h{h:02d}_Y.npy"
                if xp.exists() and yp.exists():
                    self.samples.append((xp, yp, day, h))
        gf = np.load(self.tensor_dir / "grid_features.npy")
        self.grid_features = torch.from_numpy(
            gf.transpose(2, 0, 1).astype(np.float32))
        print(f"  Dataset : {len(self.samples)} snapshots "
              f"({'aug ON' if augment else 'aug OFF'})")

    def __len__(self): return len(self.samples)

    def augment_sample(self, X, Y):
        X, Y = X.clone(), Y.clone()
        if torch.rand(1) > 0.5:
            scale = torch.FloatTensor(1).uniform_(0.8, 1.2).item()
            X[4:7] *= scale; Y *= scale
        if torch.rand(1) > 0.5:
            X[:8] += torch.randn_like(X[:8]) * 0.02
        return X, Y

    def __getitem__(self, idx):
        xp, yp, day, h = self.samples[idx]
        X = torch.from_numpy(np.load(xp).astype(np.float32).transpose(2, 0, 1))
        Y = torch.from_numpy(np.load(yp).astype(np.float32).transpose(2, 0, 1))
        if self.augment:
            X, Y = self.augment_sample(X, Y)
        X = torch.cat([X, self.grid_features], dim=0)
        _, H, W = X.shape
        sin_ch = torch.full((1,H,W), float(np.sin(2*np.pi*h/24)), dtype=torch.float32)
        cos_ch = torch.full((1,H,W), float(np.cos(2*np.pi*h/24)), dtype=torch.float32)
        X = torch.cat([X, sin_ch, cos_ch], dim=0)
        return X, Y, day, h


BATCH_SIZE    = 1
train_dataset = ANEODataset(TRAIN_DAYS, HEURES, TENSOR_DIR, augment=True)
val_dataset   = ANEODataset(VAL_DAYS,   HEURES, TENSOR_DIR, augment=False)
test_dataset  = ANEODataset(TEST_DAYS,  HEURES, TENSOR_DIR, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=1,
                          shuffle=False, num_workers=2)

X_b, Y_b, _, _ = next(iter(train_loader))
print(f"X : {X_b.shape}")
print(f"Y : {Y_b.shape}")
assert X_b.shape[1] == N_INPUT_EFFECTIVE
print("Dataloaders OK")


# ==============================================================================
# BLOC 3 — Architecture DeepONet
# ==============================================================================
class BranchNet(nn.Module):
    def __init__(self, in_channels=18, p=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 64,  kernel_size=7, stride=2, padding=3),
            nn.InstanceNorm2d(64,  affine=True), nn.GELU(),
            nn.Conv2d(64,  128, kernel_size=5, stride=2, padding=2),
            nn.InstanceNorm2d(128, affine=True), nn.GELU(),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.InstanceNorm2d(256, affine=True), nn.GELU(),
            nn.Conv2d(256, 256, kernel_size=3, stride=2, padding=1),
            nn.InstanceNorm2d(256, affine=True), nn.GELU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
        )
        self.mlp = nn.Sequential(
            nn.Linear(256, 512), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(512, 3*p),
        )

    def forward(self, x):
        return self.mlp(self.encoder(x))


class TrunkNet(nn.Module):
    def __init__(self, d_query=10, p=256, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_query, hidden), nn.Tanh(),
            nn.Linear(hidden,  hidden), nn.Tanh(),
            nn.Linear(hidden,  hidden), nn.Tanh(),
            nn.Linear(hidden,  p),
        )

    def forward(self, query):
        return self.net(query)


class DeepONet(nn.Module):
    """
    Physics-Constrained Deep Operator Network.
    G(u)(y) = somme_k branch_k(u) x trunk_k(y) + bias_k
    Branch : CNN 18ch -> 3x256 coefficients
    Trunk  : MLP 10D -> base de fonctions p-dimensionnelle
    """
    def __init__(self, in_channels=18, d_query=10, p=256, ny=500, nx=1000):
        super().__init__()
        self.p  = p
        self.ny = ny
        self.nx = nx
        self.branch  = BranchNet(in_channels=in_channels, p=p)
        self.trunk_u = TrunkNet(d_query=d_query, p=p)
        self.trunk_v = TrunkNet(d_query=d_query, p=p)
        self.trunk_w = TrunkNet(d_query=d_query, p=p)
        self.bias_u  = nn.Parameter(torch.zeros(1))
        self.bias_v  = nn.Parameter(torch.zeros(1))
        self.bias_w  = nn.Parameter(torch.zeros(1))
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
                if m.bias is not None: nn.init.zeros_(m.bias)

    def _build_query(self, x):
        B, C, H, W = x.shape
        dev = x.device
        ys = torch.linspace(0, 1, H, device=dev)
        xs = torch.linspace(0, 1, W, device=dev)
        grid_y, grid_x = torch.meshgrid(ys, xs, indexing='ij')
        coord_x = grid_x.flatten().unsqueeze(0).unsqueeze(-1).expand(B, -1, -1)
        coord_y = grid_y.flatten().unsqueeze(0).unsqueeze(-1).expand(B, -1, -1)
        terrain  = x[:, N_INPUT:N_INPUT+6, :, :].view(B, 6, -1).permute(0, 2, 1)
        hour_sin = x[:, -2:-1, :, :].reshape(B, 1, -1).permute(0, 2, 1)
        hour_cos = x[:, -1:,   :, :].reshape(B, 1, -1).permute(0, 2, 1)
        return torch.cat([coord_x, coord_y, terrain, hour_sin, hour_cos], dim=-1)

    def forward(self, x):
        B, C, H, W = x.shape
        branch_out = self.branch(x)
        b_u = branch_out[:, :self.p]
        b_v = branch_out[:, self.p:2*self.p]
        b_w = branch_out[:, 2*self.p:]
        query = self._build_query(x)
        t_u   = self.trunk_u(query)
        t_v   = self.trunk_v(query)
        t_w   = self.trunk_w(query)
        out_u = (t_u * b_u.unsqueeze(1)).sum(-1) + self.bias_u
        out_v = (t_v * b_v.unsqueeze(1)).sum(-1) + self.bias_v
        out_w = (t_w * b_w.unsqueeze(1)).sum(-1) + self.bias_w
        return torch.cat([
            out_u.view(B, 1, H, W),
            out_v.view(B, 1, H, W),
            out_w.view(B, 1, H, W),
        ], dim=1)


model = DeepONet(
    in_channels=N_INPUT_EFFECTIVE,
    d_query=10, p=256, ny=NY, nx=NX,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"DeepONet : {n_params:,} parametres  device={device}")
print("Modele OK")


# ==============================================================================
# BLOC 4 — Physics Loss
# ==============================================================================
class PhysicsConstrainedLoss(nn.Module):
    def __init__(self, lambda_mass=0.3, lambda_bc=0.05,
                 lambda_smooth=0.01, lambda_ri=0.02, dx=50.0,
                 mu_rho=1.154, std_rho=0.021):
        super().__init__()
        self.lambda_mass   = lambda_mass
        self.lambda_bc     = lambda_bc
        self.lambda_smooth = lambda_smooth
        self.lambda_ri     = lambda_ri
        self.dx            = dx
        self.register_buffer('mu_rho',  torch.tensor(mu_rho,  dtype=torch.float32))
        self.register_buffer('std_rho', torch.tensor(std_rho, dtype=torch.float32))

    def forward(self, pred, target, rho_norm, phys_weight=1.0):
        pred     = pred.float()
        target   = target.float()
        rho_norm = rho_norm.float()

        L_data   = F.mse_loss(pred, target)
        rho_phys = rho_norm * self.std_rho + self.mu_rho

        u_pred = pred[:, 0:1]
        v_pred = pred[:, 1:2]
        rho_u  = rho_phys * u_pred
        rho_v  = rho_phys * v_pred

        drhou_dx = (rho_u[:,:,:,2:] - rho_u[:,:,:,:-2]) / (2 * self.dx)
        drhov_dy = (rho_v[:,:,2:,:] - rho_v[:,:,:-2,:]) / (2 * self.dx)
        min_y = min(drhou_dx.shape[2], drhov_dy.shape[2])
        min_x = min(drhou_dx.shape[3], drhov_dy.shape[3])
        div   = drhou_dx[:,:,:min_y,:min_x] + drhov_dy[:,:,:min_y,:min_x]
        L_mass = (div**2).mean()

        w_pred = pred[:, 2:3]
        L_bc   = (w_pred[:,:,0,:]**2).mean()

        # Chaque gradient calcule separement — evite le mismatch de tailles
        L_smooth = ((pred[:,0:1,:,1:] - pred[:,0:1,:,:-1])**2).mean() + \
                   ((pred[:,0:1,1:,:] - pred[:,0:1,:-1,:])**2).mean() + \
                   ((pred[:,1:2,:,1:] - pred[:,1:2,:,:-1])**2).mean() + \
                   ((pred[:,1:2,1:,:] - pred[:,1:2,:-1,:])**2).mean()

        du_dx = (u_pred[:,:,:,1:] - u_pred[:,:,:,:-1]) / self.dx
        dv_dy = (v_pred[:,:,1:,:] - v_pred[:,:,:-1,:]) / self.dx
        L_ri  = (F.relu(du_dx**2 - 1.0).mean() +
                 F.relu(dv_dy**2 - 1.0).mean())

        phys_term = (self.lambda_mass   * L_mass +
                     self.lambda_bc     * L_bc +
                     self.lambda_smooth * L_smooth +
                     self.lambda_ri     * L_ri)

        return L_data + phys_weight * phys_term, L_data, L_mass, L_bc, L_smooth, L_ri


criterion = PhysicsConstrainedLoss(
    lambda_mass=0.3, lambda_bc=0.05,
    lambda_smooth=0.01, lambda_ri=0.02, dx=50.0,
    mu_rho=MU_RHO, std_rho=STD_RHO)
print(f"Physics loss OK  rho: mu={MU_RHO:.4f}  sigma={STD_RHO:.4f}")


# ==============================================================================
# BLOC 5 — Optimizer + OOM check
# ==============================================================================
N_EPOCHS      = 150
WARMUP_EPOCHS = 15
EARLY_STOP    = 50
ACCUM_STEPS   = 4

optimizer = torch.optim.AdamW(
    model.parameters(), lr=3e-4, weight_decay=1e-4, betas=(0.9, 0.999))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=N_EPOCHS, eta_min=1e-6)

model.train()
X_b, Y_b, _, _ = next(iter(train_loader))
X_b, Y_b = X_b.to(device), Y_b.to(device)
rho_norm  = X_b[:, 1:2, :, :]

with torch.no_grad():
    pred = model(X_b.float())
    loss, l_data, l_mass, l_bc, _, _ = criterion(
        pred, Y_b, rho_norm, phys_weight=0.0)

assert pred.shape == Y_b.shape, f"Shape: {pred.shape} vs {Y_b.shape}"
print(f"pred shape : {pred.shape}")
print(f"L_data     : {l_data.item():.6f}")
print(f"L_mass     : {l_mass.item():.6f}")
print(f"Parametres : {sum(p.numel() for p in model.parameters()):,}")
if torch.cuda.is_available():
    total_gpu = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU libre  : {total_gpu - torch.cuda.memory_reserved()/1e9:.2f} GB")
    torch.cuda.empty_cache()
print(f"N_EPOCHS={N_EPOCHS}  WARMUP={WARMUP_EPOCHS}  EARLY_STOP={EARLY_STOP}")
print("OOM check OK")


# ==============================================================================
# BLOC 6 + 7 — Fonctions + Boucle entrainement
# ==============================================================================
def physics_warmup(epoch, warmup_epochs):
    return 1.0 if warmup_epochs <= 0 else min(1.0, (epoch+1)/warmup_epochs)


def compute_r2_per_channel(pred, true):
    pred_np = pred.detach().float().cpu().numpy()
    true_np = true.detach().float().cpu().numpy()
    scores  = []
    for c in range(pred_np.shape[1]):
        y_true = true_np[:, c].flatten()
        y_pred = pred_np[:, c].flatten()
        ss_res = np.sum((y_true - y_pred)**2)
        ss_tot = np.sum((y_true - y_true.mean())**2)
        scores.append(1 - ss_res / (ss_tot + 1e-8))
    return np.mean(scores), scores


def train_epoch(epoch):
    model.train()
    train_loss = 0.0
    sum_data = sum_mass = sum_bc = sum_smooth = sum_ri = 0.0
    phys_w   = physics_warmup(epoch, WARMUP_EPOCHS)
    optimizer.zero_grad()

    for step, (X_b, Y_b, _, _) in enumerate(train_loader):
        X_b, Y_b = X_b.to(device), Y_b.to(device)
        rho_norm  = X_b[:, 1:2, :, :]
        pred = model(X_b.float())
        loss, l_data, l_mass, l_bc, l_smooth, l_ri = criterion(
            pred, Y_b, rho_norm, phys_weight=phys_w)
        (loss / ACCUM_STEPS).backward()
        if (step+1) % ACCUM_STEPS == 0 or (step+1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
        train_loss += loss.item()
        sum_data   += l_data.item()
        sum_mass   += l_mass.item()
        sum_bc     += l_bc.item()
        sum_smooth += l_smooth.item()
        sum_ri     += l_ri.item()

    n = len(train_loader)
    return (train_loss/n, sum_data/n, sum_mass/n,
            sum_bc/n, sum_smooth/n, sum_ri/n)


def validate_epoch(epoch):
    model.eval()
    val_loss = 0.0
    r2_means = []
    r2_per_ch_sum = np.zeros(3)
    phys_w   = physics_warmup(epoch, WARMUP_EPOCHS)
    sum_data = sum_mass = sum_bc = 0.0

    with torch.no_grad():
        for X_b, Y_b, _, _ in val_loader:
            X_b, Y_b = X_b.to(device), Y_b.to(device)
            rho_norm  = X_b[:, 1:2, :, :]
            pred = model(X_b.float())
            loss, l_data, l_mass, l_bc, _, _ = criterion(
                pred, Y_b, rho_norm, phys_weight=phys_w)
            val_loss  += loss.item()
            sum_data  += l_data.item()
            sum_mass  += l_mass.item()
            sum_bc    += l_bc.item()
            r2_mean, r2_ch = compute_r2_per_channel(pred, Y_b)
            r2_means.append(r2_mean)
            r2_per_ch_sum += np.array(r2_ch)

    n = len(val_loader)
    return (val_loss/n, np.mean(r2_means), r2_per_ch_sum/n,
            sum_data/n, sum_mass/n, sum_bc/n)


print("Fonctions OK")

history = {
    'train_loss':[], 'val_loss':[], 'r2':[], 'r2_u':[], 'r2_v':[], 'r2_w':[],
    'lr':[], 'phys_weight':[], 'train_l_data':[], 'train_l_mass':[],
    'train_l_bc':[], 'train_l_smooth':[], 'train_l_ri':[],
    'val_l_data':[], 'val_l_mass':[], 'val_l_bc':[],
}

best_score    = -np.inf
best_val_loss = float('inf')
best_epoch    = 0
best_r2       = -np.inf
patience_counter = 0

print("=" * 108)
print("STARTING DeepONet TRAINING")
print("  Branch(CNN) + Trunk(MLP)  rho physique  Score composite  Sans leakage")
print("=" * 108)
print(f"{'Ep':>4} | {'Train':>8} | {'Val':>8} | {'R2avg':>6} | "
      f"{'R2_u':>6} | {'R2_v':>6} | {'R2_w':>6} | "
      f"{'L_data':>7} | {'L_mass':>7} | {'ratio':>7} | "
      f"{'phi_w':>5} | {'LR':>8} | Status")
print("-" * 108)

try:
    for epoch in range(N_EPOCHS):
        (tr_loss, tr_data, tr_mass,
         tr_bc, tr_smooth, tr_ri) = train_epoch(epoch)
        (va_loss, r2, r2_ch,
         va_data, va_mass, va_bc) = validate_epoch(epoch)
        scheduler.step()

        lr         = optimizer.param_groups[0]['lr']
        phys_w     = physics_warmup(epoch, WARMUP_EPOCHS)
        mass_ratio = (0.3 * tr_mass) / (tr_data + 1e-8)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['r2'].append(r2)
        history['r2_u'].append(r2_ch[0])
        history['r2_v'].append(r2_ch[1])
        history['r2_w'].append(r2_ch[2])
        history['lr'].append(lr)
        history['phys_weight'].append(phys_w)
        history['train_l_data'].append(tr_data)
        history['train_l_mass'].append(tr_mass)
        history['train_l_bc'].append(tr_bc)
        history['train_l_smooth'].append(tr_smooth)
        history['train_l_ri'].append(tr_ri)
        history['val_l_data'].append(va_data)
        history['val_l_mass'].append(va_mass)
        history['val_l_bc'].append(va_bc)

        print(f"{epoch+1:>4} | {tr_loss:>8.5f} | {va_loss:>8.5f} | "
              f"{r2:>6.4f} | {r2_ch[0]:>6.4f} | {r2_ch[1]:>6.4f} | "
              f"{r2_ch[2]:>6.4f} | {tr_data:>7.4f} | {tr_mass:>7.4f} | "
              f"{mass_ratio:>7.4f} | {phys_w:>5.2f} | {lr:>8.2e} | ", end="")

        if phys_w >= 1.0 and mass_ratio < 0.001:
            print("PHYS FAIBLE ", end="")

        score = 0.7 * r2 + 0.3 * (-va_loss)

        if score > best_score:
            best_score    = score
            best_val_loss = va_loss
            best_epoch    = epoch + 1
            best_r2       = r2
            patience_counter = 0
            torch.save({
                'epoch': epoch+1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': va_loss, 'r2': r2,
                'r2_u': r2_ch[0], 'r2_v': r2_ch[1], 'r2_w': r2_ch[2],
                'history': history,
                'val_l_data': va_data, 'val_l_mass': va_mass,
            }, DEEPONET_MODEL_DIR / 'best_deeponet_model.pth')
            print("BEST")
        else:
            patience_counter += 1
            if patience_counter % 5 == 0:
                print(f"(patience {patience_counter}/{EARLY_STOP})")
            else:
                print("")

        if patience_counter >= EARLY_STOP:
            print(f"Early stopping epoque {epoch+1}")
            break

except KeyboardInterrupt:
    print("Interrompu")
except Exception as e:
    print(f"Echec: {type(e).__name__}: {e}")
    import traceback; traceback.print_exc()

print("=" * 108)
print("ENTRAINEMENT TERMINE")
print(f"  Meilleure epoque : {best_epoch}")
print(f"  val_loss         : {best_val_loss:.6f}")
print(f"  R2avg            : {best_r2:.4f}")
print("=" * 108)

pd.DataFrame(history).to_csv(
    OUT_DIR / 'deeponet_training_history.csv', index=False)
print("Historique sauvegarde")


# ==============================================================================
# BLOC 8 — Evaluation test set
# ==============================================================================
ckpt = torch.load(DEEPONET_MODEL_DIR / "best_deeponet_model.pth",
                  map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Charge epoque {ckpt['epoch']}  "
      f"val_loss={ckpt['val_loss']:.6f}  R2avg={ckpt['r2']:.4f}  "
      f"R2_u={ckpt['r2_u']:.4f}  R2_v={ckpt['r2_v']:.4f}  R2_w={ckpt['r2_w']:.4f}")

results = []
with torch.no_grad():
    for X_b, Y_b, day, hour in test_loader:
        day, hour = day.item(), hour.item()
        pred   = model(X_b.to(device).float())
        pred_n = pred[0].float().cpu().numpy()
        true_n = Y_b[0].numpy()

        u_p = pred_n[0]*STD_U+MU_U;  u_t = true_n[0]*STD_U+MU_U
        v_p = pred_n[1]*STD_V+MU_V;  v_t = true_n[1]*STD_V+MU_V
        w_p = pred_n[2]*STD_W+MU_W;  w_t = true_n[2]*STD_W+MU_W

        mae_u  = np.mean(np.abs(u_p-u_t))
        mae_v  = np.mean(np.abs(v_p-v_t))
        mae_w  = np.mean(np.abs(w_p-w_t))
        rmse_u = np.sqrt(np.mean((u_p-u_t)**2))
        rmse_v = np.sqrt(np.mean((v_p-v_t)**2))
        r2_u   = 1-np.sum((u_p-u_t)**2)/(np.sum((u_t-u_t.mean())**2)+1e-9)
        r2_v   = 1-np.sum((v_p-v_t)**2)/(np.sum((v_t-v_t.mean())**2)+1e-9)
        r2_w   = 1-np.sum((w_p-w_t)**2)/(np.sum((w_t-w_t.mean())**2)+1e-9)
        bias_u = np.mean(u_p-u_t)
        bias_v = np.mean(v_p-v_t)
        dir_p  = np.degrees(np.arctan2(v_p, u_p)) % 360
        dir_t  = np.degrees(np.arctan2(v_t, u_t)) % 360
        mae_dir  = np.mean(np.minimum(np.abs(dir_p-dir_t), 360-np.abs(dir_p-dir_t)))
        acc_u_05 = (np.abs(u_p-u_t) < 0.5).mean() * 100
        acc_v_05 = (np.abs(v_p-v_t) < 0.5).mean() * 100
        acc_w_01 = (np.abs(w_p-w_t) < 0.1).mean() * 100
        corr_u   = np.corrcoef(u_p.flatten(), u_t.flatten())[0, 1]
        corr_v   = np.corrcoef(v_p.flatten(), v_t.flatten())[0, 1]

        results.append(dict(
            day=day, hour=hour,
            mae_u=mae_u, mae_v=mae_v, mae_w=mae_w,
            rmse_u=rmse_u, rmse_v=rmse_v,
            r2_u=r2_u, r2_v=r2_v, r2_w=r2_w,
            bias_u=bias_u, bias_v=bias_v,
            mae_dir=mae_dir,
            acc_u_05=acc_u_05, acc_v_05=acc_v_05, acc_w_01=acc_w_01,
            corr_u=corr_u, corr_v=corr_v,
        ))
        print(f"  J{day:3d} h{hour:02d} | "
              f"R2 u={r2_u:.4f} v={r2_v:.4f} w={r2_w:.4f} | "
              f"mae_u={mae_u:.4f} | acc_u={acc_u_05:.1f}% | dir={mae_dir:.2f}deg")

df_r = pd.DataFrame(results)
df_r.to_csv(OUT_DIR / 'deeponet_evaluation_results.csv', index=False)

print("=" * 72)
print(f"TEST SET METRICS (moyenne sur {len(df_r)} snapshots):")
print("=" * 72)
for col in ['r2_u','r2_v','r2_w','mae_u','mae_v','mae_w',
            'rmse_u','rmse_v','bias_u','bias_v','mae_dir',
            'acc_u_05','acc_v_05','acc_w_01','corr_u','corr_v']:
    print(f"  {col:15s}: {df_r[col].mean():.4f}")
print("=" * 72)
print("Resultats sauvegardes -> deeponet_evaluation_results.csv")

Mounted at /content/drive
Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB
Imports OK
split.json     : False
norm_stats.csv : False
grid_features  : False
Tenseurs .npy  : 0
Recreation tenseurs depuis CSV Drive...
CSV : 30
  10/30 fichiers...
  20/30 fichiers...
  30/30 fichiers...
Jours:[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]  Heures:[0, 6, 12, 18]
Creation .npy...
  [1/30] aneo_observations_day_001_20260401_multitime.csv — 4 snapshots
  [2/30] aneo_observations_day_002_20260402_multitime.csv — 8 snapshots
  [3/30] aneo_observations_day_003_20260403_multitime.csv — 12 snapshots
  [4/30] aneo_observations_day_004_20260404_multitime.csv — 16 snapshots
  [5/30] aneo_observations_day_005_20260405_multitime.csv — 20 snapshots
  [6/30] aneo_observations_day_006_20260406_multitime.csv — 24 snapshots
  [7/30] aneo_observations_day_007_20260407_multitime.csv — 28 snapshots
  [8/30] aneo_observations_day_008_20260408_multit

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Charger l historique reel
history = pd.read_csv('/content/aneo_work/aneo_output/fno2d_training_history.csv')
results = pd.read_csv('/content/aneo_work/aneo_output/fno2d_evaluation_results.csv')

fig, axes = plt.subplots(1, 3, figsize=(15, 4), facecolor='white')

# 1. Loss curves
ep = range(1, len(history)+1)
axes[0].plot(ep, history['train_loss'], color='#94A3B8', lw=2, label='Train')
axes[0].plot(ep, history['val_loss'],   color='#2A78D6', lw=2.5, label='Val')
axes[0].axvline(history['val_loss'].idxmin()+1, color='#1BAF7A',
                lw=2, ls='--', label=f"Best ep.{history['val_loss'].idxmin()+1}")
axes[0].set_title('Loss totale', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoque'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].set_yscale('log')

# 2. R2 per channel
axes[1].plot(ep, history['r2_u'], color='#2A78D6', lw=2, label='R2_u')
axes[1].plot(ep, history['r2_v'], color='#1BAF7A', lw=2, label='R2_v')
axes[1].plot(ep, history['r2_w'], color='#EDA100', lw=2, label='R2_w')
axes[1].axhline(0.9, color='#E34948', lw=1.5, ls=':', label='Seuil 0.9')
axes[1].set_title('R2 par composante (val)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoque'); axes[1].set_ylabel('R2')
axes[1].legend()

# 3. Scatter R2 test set
u_r2 = results['r2_u'].values
v_r2 = results['r2_v'].values
w_r2 = results['r2_w'].values
x = np.arange(len(results))
axes[2].bar(x-0.25, u_r2, 0.25, label='R2_u', color='#2A78D6', alpha=0.85)
axes[2].bar(x,      v_r2, 0.25, label='R2_v', color='#1BAF7A', alpha=0.85)
axes[2].bar(x+0.25, w_r2, 0.25, label='R2_w', color='#EDA100', alpha=0.85)
axes[2].axhline(0.9, color='#E34948', lw=1.5, ls='--')
axes[2].set_title('R2 test set par snapshot', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Snapshot'); axes[2].set_ylabel('R2')
axes[2].legend(); axes[2].set_ylim(0.85, 1.0)

plt.tight_layout()
plt.savefig('/content/aneo_work/aneo_figures/figures_reelles_fno2d.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardee")

FileNotFoundError: [Errno 2] No such file or directory: '/content/aneo_work/aneo_output/fno2d_training_history.csv'